# IIR Low-Pass Filtering from Plain Instructions

A complete NumPy/SciPy implementation that generates, filters, visualizes, validates, and saves synthetic IQ data.

## Audience, prerequisites, and learning goals

**Audience:** learners familiar with Python arrays and basic IQ signals.

**Prerequisites:** NumPy indexing, the canonical `(N, 2, L)` layout, and average power.

After completing this notebook, you will be able to:

1. Build reproducible noisy IQ examples.
2. Design a second-order Butterworth low-pass IIR filter.
3. Filter I and Q independently along the time axis.
4. Compare time traces, constellations, and power.
5. save and verify a compressed NPZ artifact.

## Signal contract

- `N = 5` independent examples.
- `L = 1000` samples per example.
- Axis 1 contains I at index 0 and Q at index 1.
- Axis 2 is time and therefore the filtering axis.
- `SEED = 42` makes the signal reproducible.
- A normalized cutoff of `0.1` means 10% of the Nyquist frequency.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal as scipy_signal

## 1. Generate synthetic IQ data

Each example combines a low-frequency complex tone, a higher-frequency interferer, and Gaussian noise. The high-frequency components give the low-pass filter something measurable to attenuate.

In [ ]:
N = 5
L = 1_000
SEED = 42

rng = np.random.default_rng(SEED)
time_index = np.arange(L, dtype=np.float32)

frequencies = np.linspace(0.015, 0.035, N, dtype=np.float32)
phases = rng.uniform(0.0, 2.0 * np.pi, size=N)
low_phase = 2.0 * np.pi * frequencies[:, None] * time_index + phases[:, None]
high_phase = 2.0 * np.pi * 0.30 * time_index

I = np.cos(low_phase) + 0.35 * np.cos(high_phase) + 0.20 * rng.standard_normal((N, L))
Q = np.sin(low_phase) + 0.35 * np.sin(high_phase) + 0.20 * rng.standard_normal((N, L))
X = np.stack((I, Q), axis=1).astype(np.float32)

assert X.shape == (N, 2, L)
print(f"Input shape: {X.shape}")
print(f"Input dtype: {X.dtype}")

## 2. Design and apply the IIR filter

`scipy.signal.butter` returns the numerator and denominator coefficients. `lfilter(..., axis=2)` then applies the same causal filter independently to every example and component.

In [ ]:
ORDER = 2
CUTOFF = 0.1

b, a = scipy_signal.butter(ORDER, CUTOFF, btype="lowpass")
X_filtered = scipy_signal.lfilter(b, a, X, axis=2).astype(np.float32)

print(f"Numerator coefficients:   {b}")
print(f"Denominator coefficients: {a}")
print(f"Output shape: {X_filtered.shape}")

## 3. Compute joint IQ power

Joint power is the mean of `I² + Q²` across all examples and time samples.

In [ ]:
def compute_power(iq_array):
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return float(np.mean(i_component**2 + q_component**2))


power_before = compute_power(X)
power_after = compute_power(X_filtered)
power_ratio = power_after / power_before

print(f"Power before: {power_before:.6f}")
print(f"Power after:  {power_after:.6f}")
print(f"Power ratio:  {power_ratio:.6f}")

## 4. Compare traces, constellations, and power

The first two panels show one example in time. The third overlays decimated constellation samples, and the fourth compares global joint power.

In [ ]:
example_index = 0
trace_samples = 200
constellation_stride = 10

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(X[example_index, 0, :trace_samples], label="Before", alpha=0.75)
axes[0, 0].plot(X_filtered[example_index, 0, :trace_samples], label="After", linewidth=2)
axes[0, 0].set(title="I time trace", xlabel="Sample", ylabel="Amplitude")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(X[example_index, 1, :trace_samples], label="Before", alpha=0.75)
axes[0, 1].plot(X_filtered[example_index, 1, :trace_samples], label="After", linewidth=2)
axes[0, 1].set(title="Q time trace", xlabel="Sample", ylabel="Amplitude")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

axes[1, 0].scatter(
    X[example_index, 0, ::constellation_stride],
    X[example_index, 1, ::constellation_stride],
    s=14,
    alpha=0.45,
    label="Before",
)
axes[1, 0].scatter(
    X_filtered[example_index, 0, ::constellation_stride],
    X_filtered[example_index, 1, ::constellation_stride],
    s=14,
    alpha=0.65,
    label="After",
)
axes[1, 0].set(title="I/Q constellation", xlabel="I", ylabel="Q")
axes[1, 0].axis("equal")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

axes[1, 1].bar(["Before", "After"], [power_before, power_after], color=["tab:blue", "tab:orange"])
axes[1, 1].set(title="Joint IQ power", ylabel="Mean I² + Q²")
axes[1, 1].grid(axis="y", alpha=0.25)

fig.tight_layout()
plt.show()

## 5. Save the filtered signal

The NPZ file stores the filtered tensor, filter coefficients, and reproducibility parameters. `allow_pickle=False` keeps loading restricted to array data.

In [ ]:
OUTPUT_PATH = Path("filtered_iq.npz")
np.savez_compressed(
    OUTPUT_PATH,
    X_filtered=X_filtered,
    b=b,
    a=a,
    seed=np.array(SEED),
    cutoff=np.array(CUTOFF),
    order=np.array(ORDER),
)

with np.load(OUTPUT_PATH, allow_pickle=False) as saved:
    saved_shape = saved["X_filtered"].shape

print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Saved tensor shape: {saved_shape}")

## 6. Verification

The checks cover the required shape, dtype, finite values, changed samples, reduced power, and persisted output.

In [ ]:
assert X_filtered.shape == (5, 2, 1_000)
assert X_filtered.dtype == np.float32
assert np.isfinite(X_filtered).all()
assert not np.allclose(X_filtered, X)
assert power_after < power_before
assert 0.0 < power_ratio < 1.0
assert OUTPUT_PATH.exists()
assert saved_shape == X_filtered.shape

print("PASS: filtering, power comparison, and NPZ persistence verified.")

## Exercise

Change `CUTOFF` from `0.1` to `0.2`, redesign the filter, and predict how the constellation and power ratio will change before running it.

**Answer scaffold:** A higher cutoff keeps ______ frequency content, so the filtered constellation should become ______ and the power ratio should ______.

## Common pitfalls and extensions

- Filtering on `axis=1` would operate across the two I/Q components instead of time. ALWAYS verify the data contract before choosing an axis.
- `lfilter` is causal and begins with zero initial conditions, producing a startup transient.
- The saved path is relative to the notebook's working directory.

**Extension:** compute a frequency response with `scipy.signal.freqz` and relate its cutoff to the observed attenuation.